# MOSS BF16 Intel XPU: stage profiling

Run the same native-audio chunking, batched MOSS transcription, and speaker export workflow as `research_iGPU.ipynb`, with opt-in instrumentation. Model weights remain **BF16**, quantization is disabled, and XPU uses **math SDPA**. The worker unloads the model on success, failure, and interruption.

This notebook measures completed work by synchronizing XPU at stage and model-forward boundaries. Timings include instrumentation overhead and are **not uninstrumented production latency**. Inclusive time contains child stages; exclusive time subtracts direct children. Do not sum inclusive rows. CPU wall time, XPU kernel time, and workflow wall time are distinct quantities.

Outputs include stage CSV/JSON, per-batch input shapes and token counts, prefill/decode/encoder timing, observed KV-cache use, transcripts, native-rate speaker WAVs, and an optional CPU/XPU operator trace. Timestamps and speaker identities have the same independent-per-chunk limitations as the research notebook.

In [1]:
import sys
import json
import tempfile
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
from statistics import median
from html import escape
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "speech_to_text").is_dir():
    raise RuntimeError("Open this notebook from the project root.")
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("Select the project .venv kernel.")
from speech_to_text.chunked import transcribe_chunked
from speech_to_text.outputs import export_results
from speech_to_text.profiling import StageProfiler
print("Python:", sys.executable)


Python: /mnt/data/apps/MOSS-ASR-XPU/.venv/bin/python3


## 1. Inputs and experiments

Inputs come from `audio/active/`, with native sample rate, channels, and PCM/float precision preserved in windows and exports. Resampling/downmixing happens only in model preprocessing. Compressed inputs are decoded to WAV; their original codec/bitrate is not retained.

Change one variable at a time. Add experiment dictionaries to compare chunk duration, batch size, CPU threads, or KV cache. Every trial uses a fresh worker and one model load; the first batch includes first-use kernel overhead and is not a warmed steady-state trial. Use at least three repeats for useful medians. Compare identical inputs, prompts, token limits, trace settings, and output correctness.

In [2]:
AUDIO_DIR = PROJECT_ROOT / "audio" / "active"
AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac", ".opus", ".wma"}
audio_files = sorted(path for path in AUDIO_DIR.rglob("*")
                     if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS
                     and not path.is_symlink()
                     and path.resolve().is_relative_to(AUDIO_DIR.resolve()))
MODEL_ID = "OpenMOSS-Team/MOSS-Transcribe-Diarize"
DEVICE = "xpu:0"
MAX_NEW_TOKENS = 1024
LANGUAGES = ["Thai", "English"]
REPEATS = 1
WORKER_TIMEOUT_SECONDS = 900
MIN_AVAILABLE_RAM_GIB = 2
TRACE_BATCH = None  # Set to 1 for a separate, instrumented operator-trace run.
EXPERIMENTS = [
    {"name": "chunk_60s", "chunk_seconds": 60, "batch_size": 2, "cpu_threads": 1, "use_cache": True},
    # {"name": "baseline", "chunk_seconds": 30, "batch_size": 2, "cpu_threads": 1, "use_cache": True},
    # {"name": "chunk_10s", "chunk_seconds": 10, "batch_size": 2, "cpu_threads": 1, "use_cache": True},
]
REFERENCE_TRIAL_FILES = [PROJECT_ROOT / "outputs" / "profile-iGPU-resouzae" / "trials.json"]
output_root = PROJECT_ROOT / "outputs"
output_root.mkdir(exist_ok=True)
profile_root = Path(tempfile.mkdtemp(prefix="profile-iGPU-", dir=output_root))
print("Inputs:", [str(path.relative_to(AUDIO_DIR)) for path in audio_files])
print("Output:", profile_root)


Inputs: ['subway-test-02-thai_n_eng.wav']
Output: /mnt/data/apps/MOSS-ASR-XPU/outputs/profile-iGPU-cj_4o9gh


## 2. Execute the real BF16 workflow

The parent measures worker launch/import/exit wall time and exports. The worker measures imports, runtime initialization, weight loading, device placement, processor loading, source decoding, chunk writing, audio preprocessing, features/tokenization, padding, transfers, generation, parsing, and cleanup. Generation hooks break forward time into the first **prefill** forward and subsequent **decode** forwards, with nested audio encoder, adaptor, and language-model timings.

`prefill_forward` includes the audio encoder and the first decoder pass; it is not end-user time to first token. `decode_forward` includes full-prefix recomputation when cache is disabled. `cache_observed` checks whether a later call received a nonempty cache, rather than assuming that a config flag proves cache use.

Optional `TRACE_BATCH` records only the first two model forwards (prefill and the first decode call) of that batch with PyTorch CPU/XPU profiling, shapes, and memory tracking, to bound trace memory. This can add substantial overhead and memory usage; start with a short recording in a separate experiment. It exports a Chrome/Perfetto trace and `operators.csv`. Without a trace, stage timings are synchronized wall time, not device-kernel duration.

In [3]:
if not audio_files:
    raise ValueError("No inputs in audio/active.")
if REPEATS < 1:
    raise ValueError("REPEATS must be positive.")
trials = []
for audio_path in audio_files:
    for spec in EXPERIMENTS:
        for repeat in range(1, REPEATS + 1):
            run_dir = Path(tempfile.mkdtemp(prefix=f"{spec['name']}-{repeat}-", dir=profile_root))
            parent = StageProfiler(run_dir / "workflow")
            trial = {"source": str(audio_path), "experiment": spec, "repeat": repeat,
                     "created_at": datetime.now(timezone.utc).isoformat(),
                     "model": MODEL_ID, "device": DEVICE, "quantization": "none",
                     "trace_batch": TRACE_BATCH, "run_dir": str(run_dir)}
            started = perf_counter()
            print(audio_path.name, spec, "repeat", repeat, flush=True)
            try:
                with parent.stage("isolated_worker_launch_through_exit"):
                    raw, segments, metrics = transcribe_chunked(
                        audio_path, MODEL_ID, DEVICE, MAX_NEW_TOKENS, LANGUAGES,
                        chunk_seconds=spec["chunk_seconds"], batch_size=spec["batch_size"],
                        cpu_threads=spec["cpu_threads"], interop_threads=1,
                        quantization="none", use_cache=spec["use_cache"],
                        profile_output_dir=run_dir / "worker", trace_batch=TRACE_BATCH,
                        timeout_seconds=WORKER_TIMEOUT_SECONDS,
                        min_available_memory_bytes=MIN_AVAILABLE_RAM_GIB * 1024**3)
                with parent.stage("speaker_and_transcript_exports"):
                    export_results(audio_path, run_dir, raw, segments)
                trial.update(status="success", segment_count=len(segments), metrics=metrics)
                assert metrics["dtype"] == "bfloat16" and metrics["quantized_linear_count"] == 0
                assert metrics["allocated_bytes_after"] == metrics["baseline_allocated_bytes"]
            except BaseException as error:
                trial.update(status="failed", error=f"{type(error).__name__}: {error}")
                raise
            finally:
                trial["workflow_seconds"] = perf_counter() - started
                parent.save()
                with parent.stage("experiment_serialization"):
                    (run_dir / "experiment.json").write_text(json.dumps(trial, indent=2), encoding="utf-8")
                parent.save()
                trials.append(trial)
                (profile_root / "trials.json").write_text(json.dumps(trials, indent=2), encoding="utf-8")
            print("Completed:", trial["workflow_seconds"], "seconds;", run_dir, flush=True)


subway-test-02-thai_n_eng.wav {'name': 'chunk_60s', 'chunk_seconds': 60, 'batch_size': 2, 'cpu_threads': 1, 'use_cache': True} repeat 1


CPU threads: 1; inter-op threads: 1


Loading MOSS once on xpu:0; quantization=none


Loading weights: 100%|██████████| 683/683 [00:00<00:00, 4037.94it/s]


[transformers] `MossTranscribeDiarizeProcessor` defines `feature_extractor_class = 'AutoFeatureExtractor'`, which is deprecated. Register the correct mapping in `AutoFeatureExtractor` instead.


Batch 1: chunks 1-2/2


Batch generated tokens: [658, 177]
Batch output: 214.468s; audio 117.672s; RTF 1.823
Batch cleanup: 1817041920 bytes allocated (weights baseline: 1817041920)


Completed: 223.74006706 seconds; /mnt/data/apps/MOSS-ASR-XPU/outputs/profile-iGPU-cj_4o9gh/chunk_60s-1-bqco92uc


## 3. Rank bottlenecks by exclusive time

Choose a successful trial. The bars rank **exclusive worker stage time** so nested encoder/prefill/generation time is not counted repeatedly. `unattributed_seconds` includes gaps and instrumentation/bookkeeping outside measured scopes; worker startup before the profiler exists and process exit appear only in the parent worker wall time. Report persistence itself is not included in its own report.

Inspect inclusive times to answer higher-level questions such as total generation or total prefill latency. Inspect exclusive times for attribution. The first batch is marked cold; later batches are only subsequent batches, not guaranteed steady state.

In [4]:
successful = [trial for trial in trials if trial["status"] == "success"]
if not successful:
    raise RuntimeError("No successful trials; inspect trials.json and partial worker profiles.")
for index, trial in enumerate(successful):
    print(index, Path(trial["source"]).name, trial["experiment"]["name"], trial["repeat"])
REVIEW_INDEX = len(successful) - 1  # Inspect the latest trial by default.
selected = successful[REVIEW_INDEX]
run_dir = Path(selected["run_dir"])
profile = json.loads((run_dir / "worker" / "profile.json").read_text())
workflow = json.loads((run_dir / "workflow" / "profile.json").read_text())
rows = sorted(profile["summary"], key=lambda row: row["exclusive_seconds"], reverse=True)
scale = max(row["exclusive_seconds"] for row in rows) or 1
html_rows = "".join(
    f"<tr><td>{escape(row['stage'])}</td><td>{row['calls']}</td>"
    f"<td>{row['inclusive_seconds']:.4f}</td><td>{row['exclusive_seconds']:.4f}</td>"
    f"<td><div style='background:#4e79a7;height:14px;width:{240 * row['exclusive_seconds'] / scale:.1f}px'></div></td></tr>"
    for row in rows)
display(HTML("<table><tr><th>Stage</th><th>Calls</th><th>Inclusive (s)</th><th>Exclusive (s)</th><th>Exclusive time</th></tr>" + html_rows + "</table>"))
print("Worker unattributed seconds:", profile["unattributed_seconds"])
print("Whole workflow seconds:", selected["workflow_seconds"])
print("Parent workflow stages:", workflow["summary"])
print("Largest exclusive stage:", rows[0]["stage"])
print("Memory:", {key: value for key, value in selected["metrics"].items() if "bytes" in key})


0 subway-test-02-thai_n_eng.wav chunk_60s 1


Stage,Calls,Inclusive (s),Exclusive (s),Exclusive time
language_model,658,193.8801,193.8801,
generation,1,214.1691,13.0247,
audio_encoder,1,5.6124,5.6124,
processor_load,1,2.2911,2.2911,
model_to_device,1,1.2517,1.2517,
worker_imports,1,1.0800,1.0800,
model_weights_load,1,1.0777,1.0777,
decode_forward,657,192.0654,1.0589,
prefill_forward,1,9.0790,0.4327,
audio_read_resample_downmix,1,0.2447,0.2447,


Worker unattributed seconds: 2.1653142050017777
Whole workflow seconds: 223.74006706
Parent workflow stages: [{'stage': 'isolated_worker_launch_through_exit', 'calls': 1, 'inclusive_seconds': 223.70899473399913, 'exclusive_seconds': 223.70899473399913}, {'stage': 'speaker_and_transcript_exports', 'calls': 1, 'inclusive_seconds': 0.03066306799883023, 'exclusive_seconds': 0.03066306799883023}, {'stage': 'experiment_serialization', 'calls': 1, 'inclusive_seconds': 0.0001603739983693231, 'exclusive_seconds': 0.0001603739983693231}]
Largest exclusive stage: language_model
Memory: {'baseline_allocated_bytes': 0, 'peak_allocated_bytes': 3155883008, 'allocated_bytes_after': 0, 'reserved_bytes_after': 0, 'model_baseline_allocated_bytes': 1817041920}


## 4. Prefill, decode, cache, input shapes, and token throughput

Throughput below divides actual generated nonpadding tokens by whole-generation wall time, aggregated over the batch. Decode throughput excludes one first token per sequence because that token comes from prefill. Finished sequences may still occupy batch slots. The sums are instrumented times, with no audio download/loading/export overhead in token throughput.

A cache-enabled trial should show `cache_observed=True` when it has multiple decoding steps. Compare cache on/off separately using identical audio and inspect output text before comparing speed.

In [5]:
batch_rows = []
for batch in profile["batches"]:
    events = [event for event in profile["events"] if event.get("batch_index") == batch["index"]]
    prefill = sum(event["seconds"] for event in events if event["name"] == "prefill_forward")
    decode = sum(event["seconds"] for event in events if event["name"] == "decode_forward")
    encoder = sum(event["seconds"] for event in events if event["name"] == "audio_encoder")
    measured_batch = selected["metrics"]["batches"][batch["index"] - 1]
    token_counts = batch.get("generated_tokens", [])
    generation_events = [event for event in profile["events"] if event["name"] == "generation"]
    generation_seconds = generation_events[batch["index"] - 1]["seconds"]
    row = {"batch": batch["index"], "phase": "cold" if batch["index"] == 1 else "subsequent",
           "audio_seconds": measured_batch["audio_seconds"], "prefill_seconds": prefill,
           "decode_seconds": decode, "encoder_seconds_nested": encoder,
           "decode_calls": batch["decode_calls"], "use_cache": batch["use_cache"],
           "cache_observed": batch["cache_observed"], "tokens_per_sequence": token_counts,
           "generation_tokens_per_second": sum(token_counts) / generation_seconds,
           "decode_tokens_per_second": sum(max(0, count - 1) for count in token_counts) / decode if decode else None,
           "input_shapes": batch["input_shapes"]}
    batch_rows.append(row)
    print(json.dumps(row, indent=2))
StageProfiler.write_csv(run_dir / "batch_profile.csv", batch_rows)
print("Detailed stages:", run_dir / "worker" / "stages.csv")
print("Optional traces:", list((run_dir / "worker").glob("*trace.json")))


{
  "batch": 1,
  "phase": "cold",
  "audio_seconds": 117.672,
  "prefill_seconds": 9.079045930000575,
  "decode_seconds": 192.06537434205893,
  "encoder_seconds_nested": 5.612416502000997,
  "decode_calls": 657,
  "use_cache": true,
  "cache_observed": true,
  "tokens_per_sequence": [
    658,
    177
  ],
  "generation_tokens_per_second": 3.898788308045701,
  "decode_tokens_per_second": 4.337064933507839,
  "input_shapes": {
    "input_ids": [
      2,
      891
    ],
    "attention_mask": [
      2,
      891
    ],
    "input_features": [
      4,
      80,
      3000
    ],
    "audio_feature_lengths": [
      4
    ],
    "audio_chunk_mapping": [
      4
    ]
  }
}
Detailed stages: /mnt/data/apps/MOSS-ASR-XPU/outputs/profile-iGPU-cj_4o9gh/chunk_60s-1-bqco92uc/worker/stages.csv
Optional traces: []


## 5. Compare repeated experiments

Previously saved 30s/10s trials may be loaded through `REFERENCE_TRIAL_FILES`. They are historical single trials, not measurements collected in the same execution as the current 60s trial. Missing reference files are skipped.

Medians below include model loading, process startup/exit, and exports in workflow time; processing RTF excludes those costs. Do not mix traced and untraced runs or treat one trial as a controlled benchmark. Audio windows and padding shapes differ when chunk duration or batch size changes.

In [6]:
from collections import defaultdict
comparison = defaultdict(list)
reference_trials = []
for reference_file in REFERENCE_TRIAL_FILES:
    if reference_file.is_file():
        reference_trials.extend(trial for trial in json.loads(reference_file.read_text()) if trial["status"] == "success")
    else:
        print("Skipped missing reference:", reference_file)
for trial in reference_trials + successful:
    comparison[(trial["source"], trial["experiment"]["name"])].append(trial)
comparison_rows = []
for (source, name), group in comparison.items():
    worker_profiles = [json.loads((Path(item["run_dir"]) / "worker" / "profile.json").read_text()) for item in group]
    row = {"source": source, "experiment": name, "trials": len(group),
           "median_workflow_seconds": median(item["workflow_seconds"] for item in group),
           "median_processing_rtf": median(item["metrics"]["chunk_processing_rtf"] for item in group),
           "median_prefill_seconds": median(sum(event["seconds"] for event in profile["events"] if event["name"] == "prefill_forward") for profile in worker_profiles),
           "median_decode_seconds": median(sum(event["seconds"] for event in profile["events"] if event["name"] == "decode_forward") for profile in worker_profiles),
           "median_peak_allocated_bytes": median(item["metrics"]["peak_allocated_bytes"] for item in group),
           "median_batch_latency_seconds": median(median(batch["elapsed_seconds"] for batch in item["metrics"]["batches"]) for item in group),
           "chunk_count": group[0]["metrics"]["chunk_count"], "batch_count": group[0]["metrics"]["batch_count"]}
    comparison_rows.append(row)
    print(row)
StageProfiler.write_csv(profile_root / "comparison.csv", comparison_rows)


{'source': '/mnt/data/apps/MOSS-ASR-XPU/audio/active/subway-test-02-thai_n_eng.wav', 'experiment': 'baseline', 'trials': 1, 'median_workflow_seconds': 127.47991566600103, 'median_processing_rtf': 1.0070002190155938, 'median_prefill_seconds': 9.386357511000824, 'median_decode_seconds': 105.00711836100527, 'median_peak_allocated_bytes': 2486177280, 'median_batch_latency_seconds': 59.24786488600148, 'chunk_count': 4, 'batch_count': 2}
{'source': '/mnt/data/apps/MOSS-ASR-XPU/audio/active/subway-test-02-thai_n_eng.wav', 'experiment': 'chunk_10s', 'trials': 1, 'median_workflow_seconds': 151.52696065899727, 'median_processing_rtf': 1.2046685637874452, 'median_prefill_seconds': 19.908575674005988, 'median_decode_seconds': 117.46030417696966, 'median_peak_allocated_bytes': 2485189120, 'median_batch_latency_seconds': 24.77104400299868, 'chunk_count': 12, 'batch_count': 6}
{'source': '/mnt/data/apps/MOSS-ASR-XPU/audio/active/subway-test-02-thai_n_eng.wav', 'experiment': 'chunk_60s', 'trials': 1, 

## 6. Optimization decisions from evidence

| Priority | Method | Evidence and next experiment |
|---|---|---|
| Very high | Profiling | Rank exclusive stages; use a short operator trace to identify individual kernels/layers within the dominant module. |
| Very high | KV cache | Check observed nonempty cache and decode time; compare `use_cache=True/False` with identical inputs. |
| High | `torch.compile()` | Consider only after identifying dominant modules. Measure compilation separately from warmed inference and check graph breaks, shapes, output correctness, and memory. This notebook does not enable compilation. |
| High | OpenVINO | Requires a separately validated export/runtime integration for this remote MOSS model, audio encoder, and generation cache. No OpenVINO backend is implemented here. |
| High | Audio chunking | Compare 15/30-second windows and batch size 1/2; evaluate transcript quality and boundary/speaker errors as well as RTF. |
| Medium | Static shapes | Inspect `input_shapes` and the final partial batch; compare padding/bucketing strategies before changing shape policy. |
| Medium | Async pipeline | Consider overlapping preprocessing/transfers only if CPU preprocessing or transfer dominates. Synchronizing profiler mode prevents measuring real overlap; validate an async implementation with a separate unsynchronized throughput run and device trace. |
| Medium | Per-layer quantization | Use encoder/language-model and operator timings to choose candidate layers; preserve a BF16 reference and measure text/timestamp/speaker accuracy. No quantization is applied here. |

BF16, KV cache, and math SDPA are explicit baseline settings. `torch.compile`, OpenVINO, static padding, async execution, and selective quantization are future experiments, not hidden optimizations. Preserve the original audio parameters and unload all model/device tensors after each trial.

Save human-reviewed observations below; no WER/DER is computed without reference annotations.

In [7]:
NOTES = """Dominant stage:
KV-cache observations:
Cold versus subsequent batches:
Input-shape observations:
Transcript and speaker accuracy observations:
Next single-variable experiment:
"""
(run_dir / "optimization_notes.md").write_text(NOTES, encoding="utf-8")
print("Reports:", profile_root)


Reports: /mnt/data/apps/MOSS-ASR-XPU/outputs/profile-iGPU-cj_4o9gh
